# 06 — TaskGroup, Cancellation, and Structured Concurrency

Structured concurrency gives tasks a bounded lifetime. Use TaskGroup, timeout scopes, and cancellation-safe cleanup to avoid orphan work and confusing failure propagation.

*Advanced · 90–150 min*

## You will learn
- Coordinate related tasks in a TaskGroup
- Understand cancellation and CancelledError
- Set timeouts without losing cleanup
- Handle task failure propagation thoughtfully

## Examples

### 1. TaskGroup lifecycle

TaskGroup in Python 3.11+ waits for its child tasks before leaving the context. If a child fails with a non-cancellation exception, siblings are typically cancelled and failures are reported as an exception group.

In [ ]:
import asyncio

async def double(n):
    await asyncio.sleep(0)
    return n * 2

async def compute():
    async with asyncio.TaskGroup() as group:
        tasks = [group.create_task(double(n)) for n in (1, 2, 3)]
    return [task.result() for task in tasks]

print(asyncio.run(compute()))

**Think:** How is this lifecycle safer than dropping references to tasks?

### 2. Cancellation and finally

Cancellation usually surfaces as asyncio.CancelledError at a suspension point. Use try/finally for cleanup, and normally re-raise cancellation instead of swallowing it. The example cancels a task deterministically.

In [ ]:
import asyncio

async def job(events):
    try:
        events.append("start")
        await asyncio.sleep(10)
    finally:
        events.append("cleanup")

async def demo():
    events = []
    task = asyncio.create_task(job(events))
    await asyncio.sleep(0)
    task.cancel()
    try:
        await task
    except asyncio.CancelledError:
        events.append("cancelled")
    return events

print(asyncio.run(demo()))

**Think:** Why would hiding CancelledError potentially break structured concurrency?

### 3. Timeout policies

asyncio.timeout bounds how long an operation is allowed to run. A timeout raises TimeoutError outside the scope when the task is cancelled by the timeout. Always define whether partial work is safe to keep or must be rolled back.

In [ ]:
import asyncio

async def slow():
    await asyncio.sleep(0.1)

async def demo():
    try:
        async with asyncio.timeout(0.001):
            await slow()
    except TimeoutError:
        return "timed out"
    return "completed"

print(asyncio.run(demo()))

**Think:** What should the caller do with a partially written database transaction after timeout?

### 4. ExceptionGroup awareness

When several related tasks fail, exception groups can communicate multiple failures. except* handles matching exception types. For teaching, this small example creates a group directly; real TaskGroups construct groups based on child failures.

In [ ]:
try:
    raise ExceptionGroup("batch problems", [ValueError("bad row"), TypeError("bad type")])
except* ValueError as errors:
    print("Value problems:", len(errors.exceptions))
except* TypeError as errors:
    print("Type problems:", len(errors.exceptions))

**Think:** Why might you log every failure rather than only the first?

## Exercises

**Exercise 1.** Implement TaskGroup processing ten simulated records and return results.

In [ ]:
# Exercise 1
# TODO: Write code and tests here.

**Exercise 2.** Cancel a long-running task and demonstrate finally cleanup.

In [ ]:
# Exercise 2
# TODO: Write code and tests here.

**Exercise 3.** Wrap an async operation with timeout and return a well-defined status.

In [ ]:
# Exercise 3
# TODO: Write code and tests here.

**Exercise 4.** Demonstrate a TaskGroup child failure and explain cancellation of siblings.

In [ ]:
# Exercise 4
# TODO: Write code and tests here.

## Common mistakes

- Never suppress cancellation without an explicit design reason.
- Time bounds don't replace transactional correctness.
- TaskGroup is available in Python 3.11+.

## Quick quiz

Replace each `'?'` in the cell below with your choice and run it. The answers are hidden, but each one is checked.

**1.** When does `async with asyncio.TaskGroup()` exit?

   - **a)** After the first task finishes
   - **b)** Immediately
   - **c)** After all its child tasks finish

**2.** If one TaskGroup child fails, its siblings are…

   - **a)** Left running
   - **b)** Restarted
   - **c)** Cancelled

**3.** Swallowing CancelledError is…

   - **a)** Recommended
   - **b)** Required
   - **c)** Usually a bug: cancellation should propagate

In [ ]:
from quiz import Quiz

quiz = Quiz('advanced/06', {1: '08ee4af13cfa', 2: 'cca8d96d083d', 3: 'd7cf183d1680'})
quiz.answer(1, '?')
quiz.answer(2, '?')
quiz.answer(3, '?')

**Next:** [07 — Threads, Locks, Queues, and Shared State](07_threading_locks_queues.ipynb)